# SC-DCOPF Validation with pandapower

Independent cross-validation of the custom SC-DCOPF. The network is built from the same
data and solved with **pandapower**.

Two validations, same network:
1. **Base case** ($\mathcal{C} = \{0\}$) - pandapower's DC-OPF `rundcopp` must reproduce
   the custom optimum, **$93,152.38/h**.
2. **N-1** (config `contingencies`, `redispatch_with_exclusions`) - the custom base-case
   dispatch $P_{g,0}$ (**$109,681.25/h**) is checked **a posteriori** with pandapower's DC
   power flow `rundcpp`, one contingency at a time.

**Why a posteriori:** in the SC-DCOPF every state is coupled to the base-case dispatch
(preventive coupling for line contingencies, participation factors for generator
contingencies). pandapower cannot optimize that coupled problem (it handles one network and
one dispatch at a time), so instead of re-optimizing, we take the custom dispatch, rebuild
every post-contingency state with it and check with an independent power flow that every
limit holds. This validates **feasibility** (the dispatch really is N-1 secure); optimality
of the coupled LP is guaranteed by Gurobi.

In [ ]:
import numpy as np
import pandas as pd
import pandapower as pp

# Shared processed data lives two levels up (project/processed/<source>/).
ROOT   = ".."
SOURCE = "matpower"
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])
VN_KV     = 138.0   # single nominal voltage (cancels out in DC; fixed for consistency)
SQRT3     = np.sqrt(3.0)

print(f"base {BASE_MVA:.0f} MVA | slack {SLACK_BUS} | Vn {VN_KV} kV")
print(f"generators {len(gens)} | buses {len(loads)} | branches {len(branches)}")

## 1. Network builder

One function builds the pandapower network from the CSVs, used by both validations. Every
branch is created individually by `branch_id`, so parallel lines (same from/to) are two
independent lines — exactly as in the custom model. Returns the net plus `gen_id -> idx` and
`branch_id -> idx` lookups so we can switch elements out of service for the N-1 sweep.

In [ ]:
def build_net():
    net = pp.create_empty_network(sn_mva=BASE_MVA)

    # buses: one per MATPOWER bus number (1-indexed) -> pandapower index
    bus_idx = {}
    for b in loads["bus"].tolist():
        bus_idx[int(b)] = pp.create_bus(net, vn_kv=VN_KV, name=f"bus_{int(b)}")

    # loads: pd_pu -> MW, fixed
    for r in loads.itertuples():
        if r.pd_pu != 0.0:
            pp.create_load(net, bus=bus_idx[int(r.bus)], p_mw=r.pd_pu * BASE_MVA,
                           controllable=False, name=f"load_{int(r.bus)}")

    # generators: 19 real units; pmin/pmax pu -> MW. None is the slack as an external grid is used.
    gen_pp_idx = {}
    for r in gens.itertuples():
        gen_pp_idx[int(r.gen_id)] = pp.create_gen(
            net, bus=bus_idx[int(r.bus)], p_mw=0.0,
            min_p_mw=r.pmin_pu * BASE_MVA, max_p_mw=r.pmax_pu * BASE_MVA,
            slack=False, name=f"gen_{int(r.gen_id)}_bus_{int(r.bus)}")

    # angle reference: zero-injection ext_grid at the slack bus (fixes theta=0, injects 0 MW)
    pp.create_ext_grid(net, bus=bus_idx[SLACK_BUS], vm_pu=1.0, max_p_mw=0.0, min_p_mw=0.0)

    # branches: reactance-only impedance + MW cap calibrated so 100% loading == rate_MVA
    line_pp_idx = {}
    for r in branches.itertuples():
        rate_mva = r.rate_pu * BASE_MVA
        x_ohm    = r.x_pu * (VN_KV ** 2) / BASE_MVA
        max_i_ka = rate_mva / (SQRT3 * VN_KV)
        line_pp_idx[int(r.branch_id)] = pp.create_line_from_parameters(
            net, from_bus=bus_idx[int(r.from_bus)], to_bus=bus_idx[int(r.to_bus)],
            length_km=1.0, r_ohm_per_km=0.0, x_ohm_per_km=x_ohm, c_nf_per_km=0.0,
            max_i_ka=max_i_ka, max_loading_percent=100.0, name=f"br_{int(r.branch_id)}_{r.label}")

    # costs: linear, cp1 = c1 ($/MWh); cp0 = cp2 = 0
    for r in gens.itertuples():
        pp.create_poly_cost(net, element=gen_pp_idx[int(r.gen_id)], et="gen",
                            cp1_eur_per_mw=float(r.c1), cp0_eur=0.0)

    return net, gen_pp_idx, line_pp_idx

net0, gen_pp_idx, line_pp_idx = build_net()
print(f"builder OK: {len(net0.bus)} buses, {len(net0.gen)} gens, "
      f"{len(net0.line)} lines, {len(net0.poly_cost)} costs, {len(net0.ext_grid)} ext_grid")

## 2. Base-case validation (C = {0})

Plain DC-OPF on the full network. Should reproduce the custom base case: $93,152.38/h,
4242 MW, 13 of 19 generators dispatched, same congested lines.

In [ ]:
net, gen_pp_idx, line_pp_idx = build_net()
pp.rundcopp(net)

total_load_mw = (loads["pd_pu"] * BASE_MVA).sum()
cost_base   = float(net.res_cost)
gen_total   = net.res_gen["p_mw"].sum()
n_dispatched = int((net.res_gen["p_mw"] > 1e-3).sum())

REF_BASE = 93152.38
print(f"cost             : ${cost_base:,.2f}/h   (ref {REF_BASE:,.2f}, diff {cost_base-REF_BASE:+,.2f})")
print(f"total generation : {gen_total:,.1f} MW   (ref 4242)")
print(f"total load       : {total_load_mw:,.1f} MW")
print(f"generators > 0   : {n_dispatched} of {len(net.gen)}   (ref 13)")

In [ ]:
# dispatch per generator (merit-order check): cheap units first, expensive at zero
disp = gens.copy()
disp["pmax_mw"]   = disp["pmax_pu"] * BASE_MVA
disp["p_mw"]      = [net.res_gen.loc[gen_pp_idx[int(g)], "p_mw"] for g in disp["gen_id"]]
disp["loading_%"] = 100.0 * disp["p_mw"] / disp["pmax_mw"]
disp.sort_values("c1").reset_index(drop=True)[["gen_id", "bus", "c1", "pmax_mw", "p_mw", "loading_%"]].round(2)

In [ ]:
# congested lines (should match custom: 49-69 & 100-103 at 100%, 89-92 ~99.5%, 47-69, 94-100)
ll = branches.copy()
ll["p_from_mw"] = [net.res_line.loc[line_pp_idx[int(b)], "p_from_mw"] for b in ll["branch_id"]]
ll["loading_%"] = [net.res_line.loc[line_pp_idx[int(b)], "loading_percent"] for b in ll["branch_id"]]
ll["rate_mva"]  = ll["rate_pu"] * BASE_MVA
ll.sort_values("loading_%", ascending=False)[["branch_id", "label", "rate_mva", "p_from_mw", "loading_%"]].head(10).round(2)

## 3. N-1 validation a posteriori (config `contingencies`)

The custom base-case dispatch $P_{g,0}$ is read from the SC-DCOPF results. For each state
the generators are **fixed** to the dispatch the model implies, and a DC power flow is run:
- **base case**: $P_{g,0}$, limits $F^{\max}$;
- **line contingency** $c \in \mathcal{C}_T$: line out, $P_{g,c} = P_{g,0}$, limits $F^{\mathrm{em}}$;
- **generator contingency** $c \in \mathcal{C}_G$: unit out,
  $P_{g,c} = P_{g,0} + \alpha_{g,c} P_{h,0}$ with
  $\alpha_{g,c} = P_g^{\max} / \sum_{h} N1_{h,c} P_h^{\max}$, limits $F^{\mathrm{em}}$ and $P_g^{\max}$.

Checks per state: branch flows within the rating, bus angles within $\pm\theta^{\max}$
(the slack bus is the angle reference) and generation within $P_g^{\max}$. The ext_grid at
the slack bus has zero cost and must inject $\approx 0$ MW: the fixed dispatch closes the
power balance by itself.

The scenario set and parameters must match `sc-dcopf.ipynb`. The dispatch file is rounded
to 0.01 MW, so small violations below `TOL` are rounding.

In [ ]:
# same parameters as sc-dcopf.ipynb (keep them in sync)
EMERG_FACTOR  = 1.15         # F^em = EMERG_FACTOR * F^max
THETA_MAX     = 0.6          # bus angle limits [rad]
EXCLUDE_LINES = {7, 95}      # 8-5, 38-65
EXCLUDE_GENS  = {0}          # bus 10 (505 MW)
TOL           = 0.1          # tolerance [% of rating, MW] for the rounding of dispatch.csv

REF_COST = 109681.25
DISPATCH = f"../sc-dcopf/results/{SOURCE}/redispatch_with_exclusions/contingencies/dispatch.csv"
p0 = pd.read_csv(DISPATCH).set_index("gen_id")["p_mw"].to_dict()    # P_g0 [MW]
pmax = dict(zip(gens["gen_id"], gens["pmax_pu"] * BASE_MVA))          # P_g^max [MW]
rate = dict(zip(branches["branch_id"], branches["rate_pu"] * BASE_MVA))  # F^max [MW]

CT_lines = [int(l) for l in branches.loc[~branches["is_bridge"], "branch_id"]
            if int(l) not in EXCLUDE_LINES]
CG_gens  = [int(g) for g in gens["gen_id"] if int(g) not in EXCLUDE_GENS]
states = ([{"type": "base", "out": None}]
          + [{"type": "line", "out": l} for l in CT_lines]
          + [{"type": "gen",  "out": g} for g in CG_gens])

cost = sum(gens.set_index("gen_id").loc[g, "c1"] * p for g, p in p0.items())
print(f"dispatch: {DISPATCH}")
print(f"cost of P_g0: ${cost:,.2f}/h (ref {REF_COST:,.2f}, diff {cost - REF_COST:+.2f})")
print(f"|C| = {len(states)} states (1 base + {len(CT_lines)} lines + {len(CG_gens)} gens) - expect 194")

In [ ]:
def dispatch_for(state):
    # generator outputs [MW] the model implies in this state
    if state["type"] != "gen":
        return dict(p0)                                   # base case / preventive coupling
    h = state["out"]
    cap_on = sum(pmax[g] for g in p0 if g != h)           # sum_h N1_hc P_h^max
    return {g: (0.0 if g == h else p0[g] + pmax[g] / cap_on * p0[h]) for g in p0}

rows = []
for s in states:
    net, gidx, lidx = build_net()
    p = dispatch_for(s)
    for g, mw in p.items():
        net.gen.loc[gidx[g], "p_mw"] = mw
    if s["type"] == "line":
        net.line.loc[lidx[s["out"]], "in_service"] = False
    if s["type"] == "gen":
        net.gen.loc[gidx[s["out"]], "in_service"] = False
    pp.rundcpp(net)

    limit = 1.0 if s["type"] == "base" else EMERG_FACTOR
    load = {b: 100 * abs(net.res_line.loc[lidx[b], "p_from_mw"]) / (limit * rate[b])
            for b in lidx if not (s["type"] == "line" and b == s["out"])}
    worst = max(load, key=load.get)
    rows.append({
        "type": s["type"],
        "out": (branches.set_index("branch_id").loc[s["out"], "label"] if s["type"] == "line"
                else (f"gen bus {gens.set_index('gen_id').loc[s['out'], 'bus']}" if s["type"] == "gen" else "-")),
        "max_loading_%": load[worst],
        "worst_branch": branches.set_index("branch_id").loc[worst, "label"],
        "max_angle_rad": np.radians(net.res_bus["va_degree"].abs().max()),
        "max_gen_%": max(100 * p[g] / pmax[g] for g in p if pmax[g] > 0),
        "slack_mw": float(net.res_ext_grid["p_mw"].sum()),
    })

val = pd.DataFrame(rows)
val["ok"] = ((val["max_loading_%"] <= 100 + TOL)
             & (val["max_angle_rad"] <= THETA_MAX + 1e-6)
             & (val["max_gen_%"] <= 100 + TOL)
             & (val["slack_mw"].abs() <= TOL))
print(f"{val['ok'].sum()} of {len(val)} states pass "
      f"(max slack injection {val['slack_mw'].abs().max():.3f} MW)")
val[~val["ok"]]

In [ ]:
# summary by type, and the binding states (flow at the rating or angle at the limit)
summary = val.groupby("type", sort=False).agg(
    n=("ok", "size"), passing=("ok", "sum"),
    max_loading=("max_loading_%", "max"), max_angle=("max_angle_rad", "max"),
    max_gen=("max_gen_%", "max")).round(3)
display(summary)

binding = val[(val["max_loading_%"] >= 100 - TOL) | (val["max_angle_rad"] >= THETA_MAX - 1e-3)]
print(f"{len(binding)} binding states (these are what make the dispatch more expensive):")
binding.sort_values("max_loading_%", ascending=False).round(3)